In [ ]:
import os
import json
import matplotlib.pyplot as plt
import numpy as np

try:
    from naive import train_naive
    from warmup import train_with_warmup as train_warmup_base
    from f_warmup import train_with_warmup as train_warmup_final
    print("Execution scripts successfully imported.")
except ImportError:
    print("Import Error: Verify execution scripts are in the working directory.")

REQUIRED_LOGS = [
    'results/naive/reward_history_500000.json',
    'results/naive/win_history_500000.json',
    'results/naive/attempt_history_500000.json',
    'results/warmup/reward_history_500000.json',
    'results/warmup/win_history_500000.json',
    'results/warmup/attempt_history_500000.json',
    'results/farcat576/naive/reward_history_500000.json',
    'results/farcat576/naive/win_history_500000.json',
    'results/farcat576/naive/attempt_history_500000.json',
    'results/farcat576/f_warmup/reward_history_500000.json',
    'results/farcat576/f_warmup/win_history_500000.json',
    'results/farcat576/f_warmup/attempt_history_500000.json'
]

if any(not os.path.exists(f) for f in REQUIRED_LOGS):
    print("Logs missing. Executing training sequences...")
    train_naive()
    train_warmup_base()
    train_warmup_final()
    print("Data logging complete.")
else:
    print("All historical logs detected. Skipping training sequence.")

In [ ]:
def load_log(filepath):
    with open(filepath, 'r') as f:
        return np.array(json.load(f), dtype=np.float32)

output_dir = 'results/plots'
os.makedirs(output_dir, exist_ok=True)

reward_m1 = load_log('results/naive/reward_history_500000.json')
reward_m2 = load_log('results/warmup/reward_history_500000.json')
reward_m3 = load_log('results/farcat576/naive/reward_history_500000.json')
reward_m4 = load_log('results/farcat576/f_warmup/reward_history_500000.json')

win_m1 = load_log('results/naive/win_history_500000.json')
win_m2 = load_log('results/warmup/win_history_500000.json')
win_m3 = load_log('results/farcat576/naive/win_history_500000.json')
win_m4 = load_log('results/farcat576/f_warmup/win_history_500000.json')

attempt_m1 = load_log('results/naive/attempt_history_500000.json')
attempt_m2 = load_log('results/warmup/attempt_history_500000.json')
attempt_m3 = load_log('results/farcat576/naive/attempt_history_500000.json')
attempt_m4 = load_log('results/farcat576/f_warmup/attempt_history_500000.json')

def plot_metrics(data1, data2, data3, data4, title, ylabel, filename, window=1000):
    plt.figure(figsize=(10, 5))
    
    kernel = np.ones(window, dtype=np.float32) / window
    smoothed1 = np.convolve(data1, kernel, mode='same')
    smoothed2 = np.convolve(data2, kernel, mode='same')
    smoothed3 = np.convolve(data3, kernel, mode='same')
    smoothed4 = np.convolve(data4, kernel, mode='same')
    
    plt.plot(smoothed1[window:-window], label='Naive without CL')
    plt.plot(smoothed2[window:-window], label='Warmup without CL')
    plt.plot(smoothed3[window:-window], label='Naive with CL')
    plt.plot(smoothed4[window:-window], label='Warmup with CL')
    
    plt.title(title)
    plt.xlabel('Episodes')
    plt.ylabel(ylabel)
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)
    
    save_path = os.path.join(output_dir, filename)
    plt.savefig(save_path, bbox_inches='tight', dpi=300)
    plt.show()
    plt.close()

In [ ]:
plot_metrics(reward_m1, reward_m2, reward_m3, reward_m4, 'Reward History Comparison', 'Rewards', 'all_comparison_reward.png')
plot_metrics(win_m1, win_m2, win_m3, win_m4, 'Win Rate Comparison', 'Win Rate', 'all_comparison_winrate.png')
plot_metrics(attempt_m1, attempt_m2, attempt_m3, attempt_m4, 'Attempt History Comparison', 'Attempts', 'all_comparison_attempts.png')

In [ ]:
import os

output_dir = 'results/plots'
os.makedirs(output_dir, exist_ok=True)

def plot_pairwise(data_baseline, data_cl, label_baseline, label_cl, title, ylabel, filename, window=1000):
    plt.figure(figsize=(10, 5))
    
    kernel = np.ones(window, dtype=np.float32) / window
    smoothed_baseline = np.convolve(data_baseline, kernel, mode='same')
    smoothed_cl = np.convolve(data_cl, kernel, mode='same')
    
    plt.plot(smoothed_baseline[window:-window], label=label_baseline)
    plt.plot(smoothed_cl[window:-window], label=label_cl)
    
    plt.title(title)
    plt.xlabel('Episodes')
    plt.ylabel(ylabel)
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.6)

    save_path = os.path.join(output_dir, filename)
    plt.savefig(save_path, bbox_inches='tight', dpi=300)
    plt.show()
    plt.close()

In [ ]:
# Comparison 1: Initial Base Subsets (Naive vs Warmup Baseline)
plot_pairwise(reward_m1, reward_m2, 'Naive without CL', 'Warmup without CL', 'Base Subset: Reward Comparison', 'Rewards', 'paired_withoutCL_reward_comparison.png')
plot_pairwise(win_m1, win_m2, 'Naive without CL', 'Warmup without CL', 'Base Subset: Win Rate Comparison', 'Win Rate', 'paired_withoutCL_winrate_comparison.png')
plot_pairwise(attempt_m1, attempt_m2, 'Naive without CL', 'Warmup without CL', 'Base Subset: Attempt Comparison', 'Attempts', 'paired_withoutCL_attempt_comparison.png')

# Comparison 2: Final Global Datasets (Naive Control vs Final Curriculum Learning)
plot_pairwise(reward_m3, reward_m4, 'Naive with CL', 'Warmup with CL', 'Global Dataset: Reward Comparison', 'Rewards', 'paired_withCL_reward_comparison.png')
plot_pairwise(win_m3, win_m4, 'Naive with CL', 'Warmup with CL', 'Global Dataset: Win Rate Comparison', 'Win Rate', 'paired_withCL_winrate_comparison.png')
plot_pairwise(attempt_m3, attempt_m4, 'Naive with CL', 'Warmup with CL', 'Global Dataset: Attempt Comparison', 'Attempts', 'paired_withCL_attempt_comparison.png')